### Import Dependencies

In [ ]:
import openai
from pydantic import BaseModel, Field
import cohere

from qdrant_client import QdrantClient
from qdrant_client import models
from qdrant_client.models import Document, Prefetch, FusionQuery

### RAG Pipeline

In [3]:
qdrant_client = QdrantClient(url="http://localhost:6333")

In [4]:
def get_embedding(text, model="text-embedding-3-small"):    
    response = openai.embeddings.create(
        input=text,
        model=model,
    )
    return response.data[0].embedding

In [5]:
def retrieve_data(query, qdrant_client, k=5):

    query_embedding = get_embedding(query)

    results = qdrant_client.query_points(
        collection_name="Amazon-items-collection-01-hybrid-search",
        prefetch=[
            Prefetch(
                query=query_embedding,
                using="text-embedding-3-small",
                limit=20
            ),
            Prefetch(
                query=Document(
                    text=query,
                    model="qdrant/bm25"
                ),
                using="bm25",
                limit=20
            )
        ],
        query=FusionQuery(fusion="rrf"),
        limit=k
    )

    retrieved_context_ids = []
    retieved_context = []
    similarity_scores = []
    retrieved_context_ratings = []

    for result in results.points:
        retrieved_context_ids.append(result.payload["parent_asin"])
        retieved_context.append(result.payload["description"])
        similarity_scores.append(result.score)
        retrieved_context_ratings.append(result.payload["average_rating"])

    return {
        "retrieved_context_ids": retrieved_context_ids,
        "retrieved_context": retieved_context,
        "similarity_scores": similarity_scores,
        "retrieved_context_ratings": retrieved_context_ratings
    }

In [6]:
query = "Can I get a tablet?"

In [8]:
results = retrieve_data(query, qdrant_client, k=20)

In [9]:
results

{'retrieved_context_ids': ['B0B8NVNQKX',
  'B0B6VMB3D1',
  'B0C1NP5KYD',
  'B0BN7WWH63',
  'B0BR33XH8D',
  'B09LH466KZ',
  'B0BL2CZSHT',
  'B0B6ZZH83Y',
  'B0BFGXRMJN',
  'B0B4WCNWF3',
  'B0BVZ47N6N',
  'B0C3XYD574',
  'B0C78B1BTB',
  'B0BL12DTJ5',
  'B0BP1YXF3R',
  'B0C3LNX75K',
  'B0B965NYT9',
  'B0BRXZDBXZ',
  'B0BCQ8RJG7',
  'B09XQGN52P'],
 'retrieved_context': ['COOPERS 7 inch Kids Tablet Android 11 Tablet for Kids, 2GB RAM + 32GB ROM Toddler Tablet PC for Children, IPS Touch Screen, Dual Camera, Dual Speaker, WiFi Computer Tablet, Light Blue ✿【Good Kids tablet】This 7 inch tablet for kids with silm body and lightweight, it is easy to hold by children. Also the special design can protect the tablet well when dropping. ✿【Parental Control】Toddlers Tablet with parent mode can add or block apps. Set screen time limits. This tablet come with iwawa app. kids can get access to fun and educational games and videos. ✿【Powerful Tablets】Equipmented with quad core CPU, Android 11.0 System, 32G

### Reranking

In [7]:
cohere_client = cohere.ClientV2()

In [10]:
to_rerank = results["retrieved_context"]

In [11]:
to_rerank

['COOPERS 7 inch Kids Tablet Android 11 Tablet for Kids, 2GB RAM + 32GB ROM Toddler Tablet PC for Children, IPS Touch Screen, Dual Camera, Dual Speaker, WiFi Computer Tablet, Light Blue ✿【Good Kids tablet】This 7 inch tablet for kids with silm body and lightweight, it is easy to hold by children. Also the special design can protect the tablet well when dropping. ✿【Parental Control】Toddlers Tablet with parent mode can add or block apps. Set screen time limits. This tablet come with iwawa app. kids can get access to fun and educational games and videos. ✿【Powerful Tablets】Equipmented with quad core CPU, Android 11.0 System, 32GB Big Storage, 1024*600 IPS Screen offer a clear view. Runs Kinds of apps and game for kids smoothly. ✿【Long Lasting】Tablet for kids built with large capacity battery. For mixed use up to 6 hours. You can enjoy happy parent-child time with your kids. ✿【Best Gift】This high performance Children tablet is designed specially for kids. Best gift choice for Christmas, New

In [12]:
response = cohere_client.rerank(
    model="rerank-v4.0-pro",
    query=query,
    documents=to_rerank,
    top_n=20
)

In [13]:
response

V2RerankResponse(id='f2204b22-acd0-4a8b-b66a-bc70ff7f5632', results=[V2RerankResponseResultsItem(index=15, relevance_score=0.8442036), V2RerankResponseResultsItem(index=1, relevance_score=0.84265614), V2RerankResponseResultsItem(index=13, relevance_score=0.84265614), V2RerankResponseResultsItem(index=2, relevance_score=0.83419085), V2RerankResponseResultsItem(index=12, relevance_score=0.83419085), V2RerankResponseResultsItem(index=9, relevance_score=0.8336498), V2RerankResponseResultsItem(index=18, relevance_score=0.82480085), V2RerankResponseResultsItem(index=11, relevance_score=0.81440735), V2RerankResponseResultsItem(index=0, relevance_score=0.8090347), V2RerankResponseResultsItem(index=17, relevance_score=0.8029266), V2RerankResponseResultsItem(index=6, relevance_score=0.75750744), V2RerankResponseResultsItem(index=7, relevance_score=0.7546258), V2RerankResponseResultsItem(index=3, relevance_score=0.747321), V2RerankResponseResultsItem(index=4, relevance_score=0.73685527), V2Rerank

In [14]:
reranked_results = [to_rerank[result.index] for result in response.results]

In [15]:
reranked_results

['10.1 Inch Android Tablet, Octa-Core Processor, 2.4G 5G WiFi, 2GB RAM, 32GB ROM, Bluetooth 5.0, 5MP+13MP Dual Cameras, Android 9.0 Pie, IPS HD Display, Support GPS FM 🍒【High Performance Octa-Core Tablet】The android tablet is powered by an octa-core high-performance processor with 2.0 GHz offers fast app starts, smooth videos and excellent overall performance. The android 9.0 tablet has full access to Google Play Store, you can download the apps you like, such as Skype, YouTube, Facebook, Ins, Tik Tok, Youtube, Twitter, Gmail and more. 🍒【Full Featured 2.4/5G Wifi Tablet】This tablet is engineered 2.4GHz & 5GHz dual wifi, stay connected anytime. With the latest bluetooth 5.0, this android supports all your favorite accessories. Support GPS and FM function, you can download offline maps and use the built-in GPS sensor for navigation. 🍒【10.1 Inch Large Screen Tablet】This 10.1 inch tablet with 800x1280 resolution display widescreen, high-brightness, multi-touch screen that makes you comfort

In [16]:
response = cohere_client.rerank(
    model="rerank-v4.0-fast",
    query=query,
    documents=to_rerank,
    top_n=20
)

In [17]:
reranked_results = [to_rerank[result.index] for result in response.results]

In [18]:
reranked_results

["Fullant Kids Tablet 7 inch,Android 12 Tablet for Kids,32GB ROM 128GB Expand,Kids Software Pre-Installed,Bluetooth,Dual Camera,Toddler Tablet with Shockproof Case 【Newest Android 12 OS 】Fullant tablet for kids is equipped with the latest version of GMS-certified Android 11 GO OS, which is safer and faster than Android 10 and eliminates unwanted ads. The 1.6GHz quad-core processor is 20% faster than regular processors. And supports WiFi, Bluetooth, etc. 【Kids' Safety & Parental Controls】Fullant kids tablets can ensure kids' safety online with 2 types of parental controls. Using the Kidoz interface, parents can limit screen time, set educational goals, and filter harmful web pages. With the FamilyGroup app, you can check on your kids activity remotely at any time, including how often and how long the tablet was used, to understand your kids usage habits better. 【Learning, Playing and Reading Included】You can download various YouTube and Netflix and kids games apps from Google Play Store